In [1]:
pip install chess

     ---------------------------------------- 0.0/6.1 MB ? eta -:--:--
     ---------------------------------------- 0.0/6.1 MB 435.7 kB/s eta 0:00:15
     ---------------------------------------- 0.0/6.1 MB 393.8 kB/s eta 0:00:16
     ---------------------------------------- 0.0/6.1 MB 393.8 kB/s eta 0:00:16
     ---------------------------------------- 0.0/6.1 MB 393.8 kB/s eta 0:00:16
      --------------------------------------- 0.1/6.1 MB 504.4 kB/s eta 0:00:12
      --------------------------------------- 0.1/6.1 MB 504.4 kB/s eta 0:00:12
     - -------------------------------------- 0.2/6.1 MB 621.6 kB/s eta 0:00:10
     - -------------------------------------- 0.2/6.1 MB 621.6 kB/s eta 0:00:10
     - -------------------------------------- 0.2/6.1 MB 621.6 kB/s eta 0:00:10
     -- ------------------------------------- 0.5/6.1 MB 909.8 kB/s eta 0:00:07
     --- ------------------------------------ 0.5/6.1 MB 992.2 kB/s eta 0:00:06
     --- ------------------------------------ 0.6

## Task 1:

In [4]:
import chess
import heapq

# class to represent a node in the beam search
class SearchNode:
    def __init__(self, board, move_sequence, evaluation):
        self.board = board
        self.moves = move_sequence
        self.score = evaluation

    def __lt__(self, other):
        # higher scores are better
        return self.score > other.score

def simple_evaluation(board):
    """
    Basic material-based evaluation function.
    Positive scores favor White, negative scores favor Black.
    """
    piece_values = {
        chess.PAWN: 1,
        chess.KNIGHT: 3,
        chess.BISHOP: 3,
        chess.ROOK: 5,
        chess.QUEEN: 9,
        chess.KING: 0
    }

    score = 0
    for piece, value in piece_values.items():
        score += len(board.pieces(piece, chess.WHITE)) * value
        score -= len(board.pieces(piece, chess.BLACK)) * value
    return score

def beam_search(board, beam_width, depth_limit):
    """
    Performs beam search to find the best move sequence up to a limited depth.
    """
    initial_score = simple_evaluation(board)
    current_beam = [SearchNode(board.copy(), [], initial_score)]

    for depth in range(depth_limit):
        next_beam = []

        for node in current_beam:
            legal_moves = list(node.board.legal_moves)
            for move in legal_moves:
                new_board = node.board.copy()
                new_board.push(move)
                score = simple_evaluation(new_board)
                new_node = SearchNode(new_board, node.moves + [move], score)
                next_beam.append(new_node)

        # Keep only the top candidates
        current_beam = heapq.nlargest(beam_width, next_beam)

        if not current_beam:
            break  # No legal moves to proceed

    # Return the best path and its evaluation score
    best_node = max(current_beam, key=lambda node: node.score)
    return best_node.moves, best_node.score

if __name__ == "__main__":
    import chess.pgn

    # Start from the initial position
    board = chess.Board()
    beam_width = 3
    max_depth = 2

    best_moves, final_score = beam_search(board, beam_width, max_depth)

    print("Best move sequence:")
    for move in best_moves:
        print(board.san(move))
        board.push(move)

    print("Final evaluation score:", final_score)


Best move sequence:
Nc3
Nc6
Final evaluation score: 0


## Task 2:

In [7]:
import math
import random

# computing eculidean distance
def distance(point1, point2):
    x1, y1 = point1
    x2, y2 = point2
    return math.hypot(x2 - x1, y2 - y1)

def total_route_distance(route):
    dist = 0
    for i in range(len(route)):
        dist += distance(route[i], route[(i + 1) % len(route)])
    return dist

def swap_two(route):
    new_route = route[:]
    i, j = random.sample(range(len(route)), 2)
    new_route[i], new_route[j] = new_route[j], new_route[i]
    return new_route

# hill climbing algorithm
def hill_climb(locations, max_iterations=1000):
    current_route = locations[:]
    random.shuffle(current_route)
    current_distance = total_route_distance(current_route)

    for _ in range(max_iterations):
        candidate = swap_two(current_route)
        candidate_distance = total_route_distance(candidate)

        if candidate_distance < current_distance:
            current_route = candidate
            current_distance = candidate_distance

    return current_route, current_distance

if __name__ == "__main__":
    # Sample coordinates (x, y)
    delivery_points = [
        (0, 0),
        (2, 3),
        (5, 1),
        (6, 4),
        (8, 0),
        (1, 5)
    ]

    optimized_route, distance_covered = hill_climb(delivery_points)

    print("Optimized delivery route:")
    for point in optimized_route:
        print(point)

    print(f"\nTotal distance: {distance_covered:.2f}")


Optimized delivery route:
(2, 3)
(1, 5)
(6, 4)
(8, 0)
(5, 1)
(0, 0)

Total distance: 23.67


## Task 3:

In [10]:
import random
import math

# euclidean distance
def distance(a, b):
    return math.hypot(b[0] - a[0], b[1] - a[1])

# Total distance of the route
def route_distance(route, cities):
    total = 0
    for i in range(len(route)):
        total += distance(cities[route[i]], cities[route[(i + 1) % len(route)]])
    return total

def create_population(size, num_cities):
    population = []
    for _ in range(size):
        route = list(range(num_cities))
        random.shuffle(route)
        population.append(route)
    return population

def crossover(parent1, parent2):
    start, end = sorted(random.sample(range(len(parent1)), 2))
    child = [None] * len(parent1)

    child[start:end] = parent1[start:end]
    fill_from = [city for city in parent2 if city not in child]
    
    idx = 0
    for i in range(len(child)):
        if child[i] is None:
            child[i] = fill_from[idx]
            idx += 1

    return child

#  Swaps two cities in the route with a small probability.
def mutate(route, mutation_rate=0.02):
    new_route = route[:]
    for i in range(len(new_route)):
        if random.random() < mutation_rate:
            j = random.randint(0, len(new_route) - 1)
            new_route[i], new_route[j] = new_route[j], new_route[i]
    return new_route

# Selects the top routes based on shortest distance
def select_best(population, cities, retain_count=5):
    graded = sorted(population, key=lambda route: route_distance(route, cities))
    return graded[:retain_count]

def genetic_algorithm(cities, generations=100, pop_size=50):
    num_cities = len(cities)
    population = create_population(pop_size, num_cities)

    for gen in range(generations):
        best = select_best(population, cities)
        children = best[:]

        while len(children) < pop_size:
            parent1, parent2 = random.sample(best, 2)
            child = crossover(parent1, parent2)
            child = mutate(child)
            children.append(child)

        population = children

    best_route = min(population, key=lambda r: route_distance(r, cities))
    return best_route, route_distance(best_route, cities)

if __name__ == "__main__":
    cities = [
        (2, 3), (5, 4), (1, 7), (8, 9), (6, 2),
        (3, 8), (7, 1), (0, 4), (9, 5), (4, 0)
    ]

    route, total = genetic_algorithm(cities)
    
    print("Best route found:")
    for idx in route:
        print(cities[idx])

    print(f"\nTotal distance: {total:.2f}")


Best route found:
(8, 9)
(9, 5)
(7, 1)
(6, 2)
(5, 4)
(4, 0)
(2, 3)
(0, 4)
(1, 7)
(3, 8)

Total distance: 32.71
